> ## SOLUTION / ANSWER KEY &mdash; Lab 6.12
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-12-capstone-langchain-agent.ipynb`](../lab-12-capstone-langchain-agent.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 6.12 &mdash; Capstone: A Guardrailed LangChain Agent

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 45 min &nbsp;|&nbsp; **Day 3 &middot; Module 6 &mdash; Frameworks for Building AI Agents**

### What you'll do
- Compose the module: memory + tracing + a recursion cap over vetted tools
- Assemble allow-listed @tools into a real create_agent
- Run the real agent over a suite of tasks (optional live cell)

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`, `langgraph`). The **graded** cells assert only on the deterministic parts you build &mdash; tool wiring, prompt formatting, agent structure, routing and guardrails &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable.

**Reference:** [Module 6 slides &mdash; Choosing a framework](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 6 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-06-12"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
Capstone &mdash; a real **synthesis** of the module. You compose the pieces you built into one guarded
agent: **conversation memory** (Lab 7) so it carries context across turns, a **tracing callback**
(Lab 11) so every step is recorded, a **`recursion_limit`** cap so the loop can never run away, and an
**allow-list** so only vetted tools are bound to `create_agent`. The graded cell asserts the assembly
and that all three cross-cutting assets are actually wired (deterministic); the optional cell runs the
**real** agent (Ollama/Groq) over a task suite &mdash; the bridge to Day 4.

In [ ]:
from langchain_core.tools import tool

@tool
def lookup(key: str) -> str:
    """Look up a known fact by key."""
    return {"population of metropolis": "120000", "capital of france": "Paris"}.get(key.lower().strip(), "unknown")

print("a vetted tool:", lookup.name)

## Your Turn
Complete `build_agent` (the allow-list + `create_agent`) and `run_config` (the recursion cap), then
compose the three assets in `capstone_run`: **trace** each step (the callback) and **remember** it (the
memory), capped by the recursion limit, over the vetted tools.

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)

from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

@tool
def lookup(key: str) -> str:
    """Look up a known fact by key."""
    facts = {"population of metropolis": "120000", "capital of france": "Paris"}
    return facts.get(key.lower().strip(), "unknown")

@tool
def calculator(expression: str) -> float:
    """Do exact arithmetic on an expression."""
    return safe_calc(expression)

ALLOWED = {"lookup", "calculator"}
SYSTEM = "You are a careful assistant. Use tools for facts and math; never guess."
REGISTRY = {t.name: t for t in [lookup, calculator]}

# --- Asset 1: conversation memory (Lab 7) ---
class ConversationMemory:
    def __init__(self): self.turns = []
    def add(self, role, text): self.turns.append((role, text))
    def history(self): return "\n".join(r + ": " + t for r, t in self.turns)

# --- Asset 2: tracing callback (Lab 11) ---
class TracingCallback:
    def __init__(self): self.events = []
    def on_step(self, action, arg, obs): self.events.append((action, arg, obs))

def vet_tools(tools):
    # guardrail: keep only tools whose name is on the allow-list
    return [t for t in tools if t.name in ALLOWED]

def build_agent():
    model = ChatOllama(model="llama3.2:1b")
    tools = vet_tools([lookup, calculator])
    return create_agent(model, tools, system_prompt=SYSTEM)

# --- Asset 3: the recursion cap ---
def run_config(max_steps=8):
    return {"recursion_limit": max_steps}

def capstone_run(steps, memory, cb, max_steps=8):
    # compose all three assets over the vetted tools, capped by the recursion limit
    cap = run_config(max_steps)['recursion_limit']
    for i, (action, arg) in enumerate(steps):
        if i >= cap or action not in ALLOWED:
            break
        obs = REGISTRY[action].invoke(arg)
        cb.on_step(action, arg, obs)
        memory.add("tool", action + " -> " + str(obs))
    return {'trace': cb.events, 'history': memory.history()}

try:
    agent = build_agent()
    mem, cb = ConversationMemory(), TracingCallback()
    out = capstone_run([('lookup', 'capital of france'), ('calculator', '120000/2')], mem, cb)
    print('agent type   :', type(agent).__name__)
    print('bound tools  :', [t.name for t in vet_tools([lookup, calculator])])
    print('trace events :', out['trace'])
    print('memory       :', out['history'].replace(chr(10), ' | '))
    print('run config   :', run_config())
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("the capstone agent is a runnable CompiledStateGraph over the vetted tools", lambda: type(build_agent()).__name__ == "CompiledStateGraph")
expect_true("the guardrail drops a non-allow-listed tool", lambda: (lambda fake: [t.name for t in vet_tools([lookup, fake])] == ["lookup"])(type("F", (), {"name": "delete_database"})()))
expect_true("Asset 3: the recursion cap is set", lambda: run_config(5)["recursion_limit"] == 5)
expect_true("Asset 2: the tracing callback recorded every step", lambda: len((lambda m, c: (capstone_run([("lookup", "capital of france"), ("calculator", "120000/2")], m, c), c.events)[1])(ConversationMemory(), TracingCallback())) == 2)
expect_true("Asset 1: memory carries context across turns", lambda: (lambda m, c: (capstone_run([("lookup", "capital of france"), ("calculator", "120000/2")], m, c), m.history())[1])(ConversationMemory(), TracingCallback()).count(chr(10)) == 1)
expect_true("the recursion cap actually limits the run", lambda: len((lambda m, c: (capstone_run([("lookup", "capital of france"), ("calculator", "120000/2")], m, c, max_steps=1), c.events)[1])(ConversationMemory(), TracingCallback())) == 1)

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; run it for real (not graded)
Run the guardrailed agent on a real task -- the bridge to Day 4. (Try more questions from the SUITE list yourself.) This calls a **real** local model via `ChatOllama("llama3.2:1b")` &mdash; start it with
`ollama run llama3.2:1b` (or swap in `ChatGroq` with a `GROQ_API_KEY`). If none is reachable the cell
prints a note and moves on. **The graded cells above never call an LLM, so the lab always verifies offline.**
*(llama3.2:1b is tiny &mdash; tool-calling can be hit-or-miss; the point is to see a real invocation.)*

In [ ]:
SUITE = ["What is 15 times 3?", "What is the capital of France?", "What is the population of metropolis?"]
try:
    if ollama_up():
        agent = build_agent()
        q = SUITE[0]   # try the others yourself -- each is a fresh real agent run
        result = agent.invoke({"messages": [{"role": "user", "content": q}]}, config=run_config())
        print(q, "->", result["messages"][-1].content)
        print("That was a REAL guardrailed LangChain agent. Next: Day 4 -- task automation & multi-agent teams.")
    else:
        print("No Ollama reachable -- skipping the live run. The guardrailed agent above is fully assembled.")
        print("Next: Day 4 -- task automation & multi-agent collaboration.")
except Exception as e:
    print("Live run skipped:", type(e).__name__)

---
### Key takeaway
You assembled a guardrailed LangChain agent -- vetted tools, a system prompt, create_agent and a recursion cap -- and ran it for real. That is a shippable agent; next, Day 4 puts agents to work.

[&#8592; All Module 6 labs](./index.html) &nbsp;&middot;&nbsp; [Module 6 slides](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>